# Hermes Reasoning Tool Use - Sample Viewer

Browse formatted conversation samples from the filtered training dataset.

In [1]:
import json
import re

from IPython.display import HTML, display

DATA_PATH = "data/hermes_reasoning_tool_use_train_split_train_filtered.jsonl"

# Load all records
with open(DATA_PATH) as f:
    records = [json.loads(line) for line in f]

print(f"Loaded {len(records)} records")

Loaded 32249 records


In [2]:
from collections import Counter

cats = Counter(r["metadata"]["scenario_category"] for r in records)
sources = Counter(r["metadata"]["source"] for r in records)

print("Scenario categories:")
for k, v in cats.most_common():
    print(f"  {k}: {v}")

print("\nSources:")
for k, v in sources.most_common():
    print(f"  {k}: {v}")

Scenario categories:
  single: 14524
  multiturn: 13154
  multistep: 4571

Sources:
  ToolAce: 9170
  Nous-Hermes: 8550
  Glaive: 5585
  Nvidia-When2Call: 4755
  Salesforce-Xlam: 2252
  Xlam-Salesforce: 1937


In [3]:
ROLE_COLORS = {
    "system": "#6c757d",
    "user": "#0d6efd",
    "assistant": "#198754",
    "tool": "#dc3545",
}

ROLE_LABELS = {
    "system": "SYSTEM",
    "user": "USER",
    "assistant": "ASSISTANT",
    "tool": "TOOL",
}


def sanitize(text):
    """Remove any characters that can't round-trip through UTF-8."""
    return re.sub(r"[\ud800-\udfff]", "", text)


def format_content(content, role, max_len=2000):
    """Format message content as HTML."""
    text = sanitize(content)
    text = text if len(text) <= max_len else text[:max_len] + "\n... [truncated]"
    # Escape HTML
    text = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return f"<pre style='white-space:pre-wrap; margin:0; font-size:13px;'>{text}</pre>"


def display_conversation(record, index=None):
    """Display a single conversation with formatted turns."""
    meta = record["metadata"]
    header = f"<b>Example {index}</b>" if index is not None else "<b>Example</b>"
    header += (
        f" -- <code>{sanitize(meta['scenario_category'])}</code>"
        f" | source: <code>{sanitize(meta['source'])}</code>"
        f" | category: <code>{sanitize(meta['category'])}</code>"
    )

    html = "<div style='border:2px solid #ddd; border-radius:8px; padding:12px; margin:10px 0;'>"
    html += f"<div style='margin-bottom:10px;'>{header}</div>"

    for msg in record["messages"]:
        role = msg["role"]
        color = ROLE_COLORS.get(role, "#333")
        label = ROLE_LABELS.get(role, role.upper())
        content_html = format_content(msg["content"], role)

        html += (
            f"<div style='margin:8px 0; padding:8px; "
            f"border-left:4px solid {color}; background:#f8f9fa;'>"
            f"<div style='font-weight:bold; color:{color}; "
            f"margin-bottom:4px; font-size:14px;'>{label}</div>"
            f"{content_html}</div>"
        )

    html += "</div>"
    display(HTML(html))

## Sample from each scenario category

In [4]:
# Show 2 samples from each scenario category
shown = {}
for i, r in enumerate(records):
    cat = r["metadata"]["scenario_category"]
    if cat not in shown:
        shown[cat] = 0
    if shown[cat] < 2:
        display_conversation(r, index=i)
        shown[cat] += 1
    if all(v >= 2 for v in shown.values()):
        break

## Browse by index

In [5]:
# Change this index to browse different examples
IDX = 100
display_conversation(records[IDX], index=IDX)

## Browse by scenario category

In [6]:
# Change category and count to browse
CATEGORY = "multistep"  # single, multiturn, multistep
COUNT = 3

shown = 0
for i, r in enumerate(records):
    if r["metadata"]["scenario_category"] == CATEGORY:
        display_conversation(r, index=i)
        shown += 1
        if shown >= COUNT:
            break

## Turn statistics

In [7]:
from collections import Counter

turn_counts = Counter()
role_sequences = Counter()

for r in records:
    msgs = r["messages"]
    cat = r["metadata"]["scenario_category"]
    turn_counts[(cat, len(msgs))] += 1
    role_seq = " -> ".join(m["role"] for m in msgs)
    role_sequences[(cat, role_seq)] += 1

print("Turn counts by category:")
for (cat, n), count in sorted(turn_counts.items()):
    print(f"  {cat:12s} {n:2d} turns: {count:5d}")

print("\nMost common role sequences:")
for (cat, seq), count in role_sequences.most_common(10):
    print(f"  {cat:12s} ({count:5d}): {seq}")

Turn counts by category:
  multistep     5 turns:  1219
  multistep     6 turns:    53
  multistep     7 turns:   506
  multistep     8 turns:  2685
  multistep     9 turns:   108
  multiturn     5 turns:    15
  multiturn     7 turns:    20
  multiturn     8 turns:    39
  multiturn     9 turns:  9905
  multiturn    10 turns:     4
  multiturn    11 turns:     7
  multiturn    13 turns:  3164
  single        3 turns:  5511
  single        4 turns:  6336
  single        5 turns:  2664
  single        6 turns:    12
  single        7 turns:     1

Most common role sequences:
  multiturn    ( 9905): system -> user -> assistant -> tool -> assistant -> user -> assistant -> tool -> assistant
  single       ( 6336): system -> user -> assistant -> tool
  single       ( 5511): system -> user -> assistant
  multiturn    ( 3164): system -> user -> assistant -> tool -> assistant -> user -> assistant -> tool -> assistant -> user -> assistant -> tool -> assistant
  multistep    ( 2685): system -> u

## Data quality: assistant turns with `<tool_response>` (mislabeled)

These are training examples where an assistant turn contains `<tool_response>` instead of
`<tool_call>`. This teaches the model to hallucinate tool responses.

In [8]:
from collections import Counter

# Find all records where an assistant turn has <tool_response> but no <tool_call>
bad_records = []
for i, r in enumerate(records):
    for j, m in enumerate(r["messages"]):
        if (
            m["role"] == "assistant"
            and "<tool_response>" in m["content"]
            and "<tool_call>" not in m["content"]
        ):
            bad_records.append((i, j, r))
            break

bad_by_source = Counter(r["metadata"]["source"] for _, _, r in bad_records)
bad_by_cat = Counter(r["metadata"]["scenario_category"] for _, _, r in bad_records)

print(f"Records with mislabeled assistant turns: {len(bad_records)} / {len(records)}")
print()
print("By source:")
for k, v in bad_by_source.most_common():
    print(f"  {k}: {v}")
print()
print("By category:")
for k, v in bad_by_cat.most_common():
    print(f"  {k}: {v}")

Records with mislabeled assistant turns: 310 / 32249

By source:
  ToolAce: 197
  Glaive: 64
  Nous-Hermes: 49

By category:
  multistep: 168
  multiturn: 142


In [10]:
# Display mislabeled examples with the bad assistant turn highlighted
# The bad turn is highlighted with a red background


def display_conversation_highlighted(record, bad_msg_idx, index=None):
    """Display conversation with the problematic assistant turn highlighted."""
    meta = record["metadata"]
    header = f"<b>Example {index}</b>" if index is not None else "<b>Example</b>"
    header += (
        f" -- <code>{sanitize(meta['scenario_category'])}</code>"
        f" | source: <code>{sanitize(meta['source'])}</code>"
        f" | category: <code>{sanitize(meta['category'])}</code>"
    )

    html = "<div style='border:2px solid #dc3545; border-radius:8px; padding:12px; margin:10px 0;'>"
    html += f"<div style='margin-bottom:10px;'>{header}</div>"

    for j, msg in enumerate(record["messages"]):
        role = msg["role"]
        color = ROLE_COLORS.get(role, "#333")
        label = ROLE_LABELS.get(role, role.upper())

        if j == bad_msg_idx:
            bg = "#f8d7da"
            label = "ASSISTANT (MISLABELED - contains tool_response)"
            color = "#dc3545"
        else:
            bg = "#f8f9fa"

        content_html = format_content(msg["content"], role)
        html += (
            f"<div style='margin:8px 0; padding:8px; "
            f"border-left:4px solid {color}; background:{bg};'>"
            f"<div style='font-weight:bold; color:{color}; "
            f"margin-bottom:4px; font-size:14px;'>{label}</div>"
            f"{content_html}</div>"
        )

    html += "</div>"
    display(HTML(html))


# Show first 5 mislabeled examples
COUNT = 5
for idx, (rec_idx, msg_idx, rec) in enumerate(bad_records[:COUNT]):
    display_conversation_highlighted(rec, msg_idx, index=rec_idx)

In [11]:
# Browse more mislabeled examples by source
SOURCE = "ToolAce"  # ToolAce, Glaive, Nous-Hermes
COUNT = 3

filtered_bad = [
    (i, j, r) for i, j, r in bad_records if r["metadata"]["source"] == SOURCE
]
print(f"{len(filtered_bad)} mislabeled records from {SOURCE}")

for idx, (rec_idx, msg_idx, rec) in enumerate(filtered_bad[:COUNT]):
    display_conversation_highlighted(rec, msg_idx, index=rec_idx)

197 mislabeled records from ToolAce
